# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeremOzcn/flyrank-ml-internship-submission/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton outlines the research question, decision framing, data grounding, and claims boundary for the FlyRank ML Internship track.

## 1. My lane (or freestyle) and why

**Chosen Lane**: **Lane 2 — Refresh / Content Opportunity Scoring**

**Why this lane?** Content decay is one of the single largest causes of organic search traffic loss across client portfolios. Content teams cannot manually inspect tens of thousands of pages every month to determine which ones need updating. Lane 2 directly addresses this high-impact operational bottleneck by building a transparent, prioritized review queue. It combines clear observable signals (impressions, age, freshness, CTR, and engagement) to identify which declining or high-opportunity pages yield the highest return on editorial review time.

In [3]:
# Verification code: confirm starter dataset availability
import os
data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv"
print("Data file confirmed at:", os.path.abspath(data_path))


Data file confirmed at: /Users/keremozcan/Desktop/flyrank_intern/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

### Decision & Unit of Analysis
- **Unit of Analysis**: A single pseudonymized content page (`content_id`) for a given client (`client_id`).
- **Decision**: *"Which content pages should the editorial/SEO team prioritize for content refresh, expansion, or protection review first?"*

### Who acts and what action is taken?
- **Actor**: Content Editors, SEO Strategists, and Digital Publishers.
- **Action**: Reviewing the ranked opportunity queue, reading clear reason codes (e.g. `stale_visible_page`, `declining_with_demand`), and executing targeted content refreshes (updating statistics, expanding thin sections, or fixing metadata).

### Cost of a Wrong Call
- **False Positive (recommending a healthy/stable page for refresh)**: Wastes 2–4 hours of valuable editorial budget on unnecessary updates.
- **False Negative (missing a high-demand page in active traffic decay)**: Results in compounding loss of organic search visibility, clicks, and client revenue.

### Why Data/ML helps over plain rules
A static human rule (e.g. `days_since_update > 180`) catches obvious stale pages but fails to handle non-linear interactions across traffic demand, position decay, CTR drops, and user engagement signals. A learned model dynamically ranks pages by combining multi-signal patterns, while remaining transparent through reason codes.

In [5]:
# Code check: verify unit of analysis (content_id) grain
import pandas as pd, os
data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv"
df_sample = pd.read_csv(data_path, usecols=["content_id", "client_id"])
print(f"Total rows: {len(df_sample):,} | Unique content IDs: {df_sample['content_id'].nunique():,}")
assert len(df_sample) == df_sample['content_id'].nunique(), "Grain check failed: duplicate content_id found!"
print("Grain verified: 1 row = 1 unique content_id.")


Total rows: 30,000 | Unique content IDs: 30,000
Grain verified: 1 row = 1 unique content_id.


## 3. Quick look at the data (2-3 real numbers)

To confirm that **Lane 2 (Content Opportunity Scoring)** is worth focusing on for the next 7 weeks, we calculate three key grounding metrics directly from the 30,000-row starter dataset below.

In [7]:
import pandas as pd, numpy as np, os

data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
total_pages = len(df)

# Number 1: Base rate of declining pages
declining_count = (df["trend_direction"] == "down").sum()
declining_rate = declining_count / total_pages

# Number 2: High-opportunity stale and visible pages
stale_visible_mask = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 100)
stale_visible_count = stale_visible_mask.sum()
stale_visible_rate = stale_visible_count / total_pages

# Number 3: Impression exposure concentration (Top 10% pages share of total impressions)
top10_cutoff = int(total_pages * 0.10)
top10_impressions = df["impressions_90d"].sort_values(ascending=False).iloc[:top10_cutoff].sum()
total_impressions = df["impressions_90d"].sum()
impression_concentration = top10_impressions / total_impressions

print(f"=== REAL DATA GROUNDING NUMBERS (Starter Dataset: {total_pages:,} rows) ===")
print(f"1. Base Rate of Declining Pages: {declining_rate:.3f} ({declining_rate*100:.1f}% | {declining_count:,} pages)")
print(f"2. Stale (>=90d update) & Visible (>=100 imp) Pages: {stale_visible_count:,} pages ({stale_visible_rate*100:.1f}%)")
print(f"3. Impression Concentration (Top 10% pages): {impression_concentration*100:.1f}% of all 90-day impressions")


=== REAL DATA GROUNDING NUMBERS (Starter Dataset: 30,000 rows) ===
1. Base Rate of Declining Pages: 0.542 (54.2% | 16,262 pages)
2. Stale (>=90d update) & Visible (>=100 imp) Pages: 8,119 pages (27.1%)
3. Impression Concentration (Top 10% pages): 70.2% of all 90-day impressions


## 4. Careful words: what I can and can't claim

### What I CAN claim:
- **Observed Associations**: Statistical relationships between observable signals (e.g. content age, CTR tiers, impression volume) and traffic movement.
- **Decision-Support Prioritization**: Demonstrating that a learned model or hybrid baseline ranks declining pages more accurately (higher Precision@K) than random selection or simple single-metric rules.
- **Client-Grouped Validation**: Proving that ranking performance generalizes across held-out client portfolios.

### What I CANNOT claim:
- **Causal Proof**: I will never claim that executing a refresh *causes* ranking or traffic recovery (which would require randomized experiments or A/B testing).
- **'Reverse-Engineering Google'**: I will never claim to predict Google's search algorithm factors.
- **Private/Client Details**: No raw URLs, client names, or un-anonymized queries will ever be published or claimed.

In [9]:
# Code check: verify public safety compliance on columns
import pandas as pd, os
data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path, nrows=5)
unsafe_terms = ['client_name', 'url', 'domain', 'raw_query', 'email']
found_unsafe = [c for c in df.columns if any(u in c.lower() for u in unsafe_terms)]
print("Unsafe raw column check:", "CLEAN (No raw identity columns found)" if not found_unsafe else f"WARNING: {found_unsafe}")


Unsafe raw column check: CLEAN (No raw identity columns found)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.